<a href="https://colab.research.google.com/github/oliversebastianmartinesdiaz-cmyk/Applied-Artificial-Intelligence-Course-with-Llama/blob/main/Challenge_3_On_Fine_Tuning_con_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1cKZ_hCf231RE84FDvGkEiKMH6ZDkVZzH?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [ ]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('API_N_1'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.1 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [ ]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [ ]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo
def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()
prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"
respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No, no puedes cambiar tu pedido.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [ ]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [ ]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()
# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [ ]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Parameter 'fn_kwargs'={'processing_class': LlamaTokenizer(name_or_path='TinyLlama/TinyLlama-1.1B-Chat-v1.0', vocab_size=32000, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), 'dataset_text_field': 'texto', 'assistant_only_loss': False, 'chat_template': None} of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, th

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '3.527', 'grad_norm': '1.896', 'learning_rate': '0.0002', 'entropy': '2.584', 'num_tokens': '367', 'mean_token_accuracy': '0.3923', 'epoch': '1'}
{'loss': '3.488', 'grad_norm': '1.769', 'learning_rate': '0.0001933', 'entropy': '2.581', 'num_tokens': '734', 'mean_token_accuracy': '0.3978', 'epoch': '2'}
{'loss': '3.433', 'grad_norm': '1.731', 'learning_rate': '0.0001867', 'entropy': '2.575', 'num_tokens': '1101', 'mean_token_accuracy': '0.3978', 'epoch': '3'}
{'loss': '3.372', 'grad_norm': '1.746', 'learning_rate': '0.00018', 'entropy': '2.579', 'num_tokens': '1468', 'mean_token_accuracy': '0.4006', 'epoch': '4'}
{'loss': '3.302', 'grad_norm': '1.758', 'learning_rate': '0.0001733', 'entropy': '2.591', 'num_tokens': '1835', 'mean_token_accuracy': '0.4033', 'epoch': '5'}
{'loss': '3.234', 'grad_norm': '1.759', 'learning_rate': '0.0001667', 'entropy': '2.609', 'num_tokens': '2202', 'mean_token_accuracy': '0.4116', 'epoch': '6'}
{'loss': '3.169', 'grad_norm': '1.762', 'learning_rat

### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.

In [ ]:
# Comparar la pérdidas
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,
# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 3.53
Pérdida final del entrenamiento: 2.74
Reducción: 22%


In [ ]:
# Referencia cualitativa (variación de sesión a sesión con un dataset chico)
# Referencia cualitativa (variación de sesión a sesión con un dataset chico)

respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
print("\nRespuesta del modelo ajustado (referencia):\n", respuesta_ajustada)


Respuesta del modelo ajustado (referencia):
 Resp>


**Nota:** el texto generado por el modelo ajustado puede variar de una ejecución a otra — con un dataset de solo 5 ejemplos y un learning rate alto (pensado para que el modelo aprenda rápido en pocos minutos), a veces la respuesta sale coherente y a veces sale con ruido. Esto es esperable en una demo de este tamaño, no un fallo del fine-tuning. La evidencia real de que el modelo aprendió es la **reducción de pérdida** (`perdida_inicial` vs. `perdida_final`), no el texto en sí — esa métrica sí es consistente ejecución tras ejecución, y es el criterio objetivo que estamos comprobando.

## **CHALLENGE: AJUSTE DE TONO CON LORA**

Una vez visto el ***Hands-On: Fine-tuning con LoRA***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se ajustará un modelo Llama ligero con un dataset propio para enseñarle un tono o formato de respuesta específico, comparando la pérdida **antes** y **después** del fine-tuning como métrica objetiva. En esta solución se usa como ejemplo un asistente de dudas frecuentes del propio curso.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Carga el modelo y define tu dataset:**

   * Instala las librerías, inicia sesión en Hugging Face con tu token y carga el modelo base junto con la función `generar_respuesta`.

   * Construye una lista llamada `ejemplos` con al menos 4 pares de entrada/respuesta que reflejen el tono o formato que quieres enseñarle al modelo, y conviértela en `dataset`.

In [ ]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('API_N_1'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.1 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.


In [ ]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
# Definir la funcion de generacion de texto
def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )

    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

In [ ]:
# Definir la lista ejemplos y convertirla en dataset
# Celda 36: Definir dataset con tono técnico y directo
from datasets import Dataset

ejemplos = [
    {"texto": "Usuario: ¿Por qué mi motorreductor pierde fuerza al subir voltaje?\nAsistente: [ROBOT-SOPORTE]: Probable saturación térmica o caída de corriente en el puente H; verifica el consumo en amperios."},
    {"texto": "Usuario: ¿Qué sensor es mejor para detectar la línea blanca del dohyo?\nAsistente: [ROBOT-SOPORTE]: Sensores ópticos infrarrojos por reflexión tipo TCRT5000 o QTR-1A calibrados por umbral digital."},
    {"texto": "Usuario: ¿Puedo alimentar el microcontrolador y los motores con la misma batería LiPo?\nAsistente: [ROBOT-SOPORTE]: Sí, siempre que uses reguladores de voltaje independientes y desacoples ruido con capacitores."},
    {"texto": "Usuario: ¿Cómo evito que el robot patine al arrancar en el combate?\nAsistente: [ROBOT-SOPORTE]: Usa llantas de silicón vulcanizado o poliuretano de durómetro bajo (30A a 40A) con rampa de aceleración por software."},
    {"texto": "Usuario: ¿Cuál es el tiempo de espera reglamentario antes de iniciar el combate?\nAsistente: [ROBOT-SOPORTE]: Obligatoriamente 5 segundos tras presionar el botón de inicio antes de cualquier movimiento."},
]

dataset = Dataset.from_list(ejemplos)
print("Dataset creado con éxito. Número de ejemplos:", len(dataset))

Dataset creado con éxito. Número de ejemplos: 5


**2. Prueba el modelo base:** Genera una respuesta con el modelo sin ajustar para un prompt de prueba y guárdala en `respuesta_base`.

In [ ]:
# Probar el modelo base con un prompt de prueba y guardar el resultado en respuesta_base
prompt_prueba = "Usuario: ¿Cómo evito que el robot patine al arrancar en el combate?\nAsistente:"
respuesta_base = generar_respuesta(modelo, prompt_prueba)
print("--- RESPUESTA DEL MODELO BASE (SIN ENTRENAR) ---")
print(respuesta_base)

--- RESPUESTA DEL MODELO BASE (SIN ENTRENAR) ---
¿Por qué no? El robot patín en el arranque.


**3. Configura y aplica LoRA:** Fija una semilla con `set_seed` y define tu `LoraConfig` (rango, alpha, módulos objetivo, dropout en 0) y aplícalo al modelo base.

In [ ]:
# Configurar LoraConfig y aplicarlo al modelo base
!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


**4. Entrena:** Configura el `SFTTrainer` con tu dataset y ejecuta el fine-tuning; guarda la pérdida final en `perdida_final`.

In [ ]:
# Configurar el Trainer con el dataset y ejecutar el fine-tuning; guardar la pérdida final en perdida_final

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_challenge",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
perdida_final = resultado_entrenamiento.training_loss

print("\nEntrenamiento completado.")
print("Pérdida final:", perdida_final)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '3.527', 'grad_norm': '1.896', 'learning_rate': '0.0002', 'entropy': '2.584', 'num_tokens': '367', 'mean_token_accuracy': '0.3923', 'epoch': '1'}
{'loss': '3.488', 'grad_norm': '1.769', 'learning_rate': '0.0001933', 'entropy': '2.581', 'num_tokens': '734', 'mean_token_accuracy': '0.3978', 'epoch': '2'}
{'loss': '3.433', 'grad_norm': '1.731', 'learning_rate': '0.0001867', 'entropy': '2.575', 'num_tokens': '1101', 'mean_token_accuracy': '0.3978', 'epoch': '3'}
{'loss': '3.372', 'grad_norm': '1.746', 'learning_rate': '0.00018', 'entropy': '2.579', 'num_tokens': '1468', 'mean_token_accuracy': '0.4006', 'epoch': '4'}
{'loss': '3.302', 'grad_norm': '1.758', 'learning_rate': '0.0001733', 'entropy': '2.591', 'num_tokens': '1835', 'mean_token_accuracy': '0.4033', 'epoch': '5'}
{'loss': '3.234', 'grad_norm': '1.759', 'learning_rate': '0.0001667', 'entropy': '2.609', 'num_tokens': '2202', 'mean_token_accuracy': '0.4116', 'epoch': '6'}
{'loss': '3.169', 'grad_norm': '1.762', 'learning_rat

**5. Compara y concluye:** Calcula la reducción entre la pérdida inicial y `perdida_final` como métrica objetiva, y usa la respuesta generada por el modelo ya ajustado solo como referencia cualitativa.

In [ ]:
# Comparar la perdida inicial y final del entrenamiento como metrica objetiva
perdida_inicial = trainer.state.log_history[0]['loss']

print(f"Pérdida inicial: {perdida_inicial:.2f}")
print(f"Pérdida final:   {perdida_final:.2f}")

reduccion = (1 - (perdida_final / perdida_inicial)) * 100
print(f"Reducción del error: {reduccion:.1f}%")

Pérdida inicial: 3.53
Pérdida final:   2.74
Reducción del error: 22.3%


In [ ]:
# Como referencia cualitativa (puede variar de sesión a sesión con un dataset tan chico):
respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)

print("--- ANTES DEL FINE-TUNING ---")
print(respuesta_base)

print("\n--- DESPUÉS DEL FINE-TUNING (CON LORA) ---")
print(respuesta_ajustada)

--- ANTES DEL FINE-TUNING ---
¿Por qué no? El robot patín en el arranque.

--- DESPUÉS DEL FINE-TUNING (CON LORA) ---
Resp>
